## Objective

* The objective of this project is to demonstarte your expertise to potential employers
* Employers are looking for people "who can hit the ground running" 

* What this means is that they want someone they can trust to get the right data to the right users
* While showcasing technical chops are great, you can put yourself at the top of their list by showing you can get the data to end users, on time and correctly
* Demonstrate this by showing
  - Who your end users are
  - What data & metrics do they care about
  - How you care about your end users by implementing data quality
  - How you think about end-user experiences by defining data contracts
  - How you enable your team with clean code and documentation
* In the following chapters we will work backwards from *outcomes -> architecture -> implementation -> visualization* and final create a presentable project that you can use to demonstarte your expertise.

## Define Outcomes

* Let's start with defining who our stakeholders are and what questions they are looking to answer.
* For this capstone project we assume that we are working with ads data on a website. 

* We track user activity (impressions, clicks, conversions) and run ad campaigns.

```text
advertiser
  └── campaign
        └── ad_group
              └── ad_creative   
```
* **Note** The data is auto-generated by a fake data generator that I created for this course, [see details here](../../../datagen/README.html)
* Let's define our stakeholders, the questions that they want answers to and the ad industry-standard metrics that they use


| Stakeholder | Core Question | Key Metrics |
|---|---|---|
| **Marketing Agents** | Is my spend efficient, and where is the funnel leaking? | ROAS, CVR, CPA, CTR, rolling windows CVR |

* Let's define the key metrics that we need to produce for our stakeholders
* If you have worked in advertisement industry these metrics will be very familiar.

| Metric | Definition | Formula |
|---|---|---|
| **ROAS** | Return on ad spend — revenue generated per $ spent | `total_revenue / total_spend` |
| **CTR** | Click-through rate — % of impressions that result in a click | `total_clicks / total_impressions` |
| **CVR** | Conversion rate — % of clicks that result in a purchase | `total_conversions / total_clicks` |
| **CPA** | Cost per acquisition — ad spend per conversion | `total_spend / total_conversions` |
| **l7d / l30d / l90d CVR** | Rolling window aggregation over last 7, 30, 90 days | `SUM(CVR) over preceding N days` |

* Additional [SAAS revenue metrics reference](https://posthog.com/handbook/product/metrics#metrics-we-use-in-growth-reviews)

## Architecture & Data Flow

* We know who our stakeholders are, the questions that they are looking to answer and the metrics that they use to answer those questions.

* With architecture try to stick with industry standard.

* Using industry standards will reduce the cognitive load for potential employer to get to the conclusion that you know what you are doing.

* With this in mind a multi hop architecture is almost always a good idea.
* **Principles**: Here is why a mutli-hop architecture makes sense
    - `Separation of concerns`: raw ingestion, analytics-ready, and aggregation are independent layers
    - `Schema evolution safety`: new Postgres columns don't break downstream bronze tables
    - `Replay without source`: Bronze decouples Silver/Gold from source availability
* **Note** call this medallion architecture to align with market/naming trends
* With these principles in mind, lets define the architecture
* We will use the standard Bronze -> Silver -> Gold (OBT & Summary) naming conventions of medallion
* Let's look at what our tables will look like
```mermaid
flowchart LR
    subgraph PG["Source-Postgres"]
        p1[advertiser]
        p2[campaign]
        p3[ad_group]
        p4[ad_creative]
        p5[ad_impression]
        p6[ad_click]
        p7[ad_conversion]
    end
    subgraph BR["Bronze"]
        b1[advertiser]
        b2[campaign]
        b3[ad_group]
        b4[ad_creative]
        b5[ad_impression]
        b6[ad_click]
        b7[ad_conversion]
    end
    subgraph SL["Silver"]
        s1[dim_advertiser]
        s2[dim_campaign]
        s9[fct_ad_impressions]
        s10[fct_ad_clicks]
        s11[fct_ad_conversions]
    end
    subgraph GD["Gold"]
        g2[obt_ad_impressions]
        g4[ad_campaign_performance]
        g5[ad_funnel_summary]
    end
    p1 --> b1 --> s1
    p2 --> b2 --> s2
    p3 --> b3 --> s2
    p4 --> b4 --> s9
    p5 --> b5 --> s9
    p6 --> b6 --> s10
    p7 --> b7 --> s11
    s1 --> g2
    s2 --> g2
    s9 --> g2
    s10 --> g2
    s11 --> g2
    g2 --> g4
    g2 --> g5

    classDef bronze fill:#7c3f00,stroke:#5a2d00,color:#fff
    classDef silver fill:#c0c0c0,stroke:#a8a8a8,color:#333
    classDef gold fill:#ffd700,stroke:#daa520,color:#333

    class b1,b2,b3,b4,b5,b6,b7 bronze
    class s1,s2,s3,s9,s10,s11 silver
    class g2,g4,g5 gold

```

## Write code

* With the 3 layers and their tables defined, its time to write the code to create these tables.
* Lets create one script per table and organize them into folder to make it easy for anyone to easily understand the flow

### Bronze Pipeline Scripts

#### Exercise [30 min]
* Create the folders 

- `/capstone_project/etl/bronze`

- `./capstone_project/etl/silver`

- `./capstone_project/etl/gold`

* Create python scripts for the respective tables (see the data flow above) in their corresponding layers.

* Now let's fill in the code for the `bronze` layer.

* Let's take a look at a bronze layer table `advertiser.`

In [ ]:
%%bash
mkdir -p ./capstone_project/bronze
mkdir -p ./capstone_project/silver
mkdir -p ./capstone_project/gold

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.bronze.advertiser"


def run(spark: SparkSession) -> None:
    spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.advertiser
        ) advertiser""",
        properties=JDBC_PROPERTIES,
    ).writeTo(TABLE_NAME).createOrReplace()


"""
if __name__ == "__main__":
    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    run(spark)
"""

* Assume: Retention policy: Handled by Iceberg history

* Alternatively, partition by loaded_at (based on pipeline run time) and in silver read as `select * from bronze_table where loaded_at = (select max(loaded_at) from bronze_table)`

* Bronze exists even for append-only fact source data.
  - Decouples Silver from Postgres availability
  - Enables full pipeline replay without re-querying Postgres
  - Raw audit log for debugging upstream

In [ ]:
# Start a SparkSession
from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("08_capstone_project").master("local[*]").getOrCreate()
)

In [ ]:
%%bash 
%%capture
uv run ./capstone_project/bronze/ad_click.py 
uv run ./capstone_project/bronze/ad_conversion.py
uv run ./capstone_project/bronze/ad_impression.py
uv run ./capstone_project/bronze/ad_creative.py 
uv run ./capstone_project/bronze/ad_group.py 
uv run ./capstone_project/bronze/advertiser.py 
uv run ./capstone_project/bronze/campaign.py 

In [ ]:
from pyspark.sql import Row, SparkSession

# Create Spark session
spark = (
    SparkSession.builder.appName("capstone_project").master("local[*]").getOrCreate()
)

In [ ]:
for table_name in [
    "ad_click",
    "ad_conversion",
    "ad_impression",
    "ad_creative",
    "ad_group",
    "advertiser",
    "campaign",
]:
    display(spark.table(f"local.bronze.{table_name}").limit(2).toPandas())

In [ ]:
spark.sql("select * from bronze.ad_click.snapshots").limit(5).toPandas()

### Silver Pipeline Scripts

* For the silver layer, lets start with the dimension tables.

#### Example

* Let's look at the script to create `dim_advertiser` below

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.silver.dim_advertiser"


def extract(spark: SparkSession) -> dict[str, DataFrame]:
    return {"advertiser": spark.table("local.bronze.advertiser")}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    return input_dfs["advertiser"].select(
        F.col("advertiser_id"),
        F.col("name").alias("advertiser_name"),
        F.col("billing_email"),
        F.col("status"),
        (F.col("status") == "active").alias("is_active"),
        F.col("created_at"),
        F.col("updated_at"),
    )


def load(output_df: DataFrame) -> None:
    output_df.writeTo(TABLE_NAME).createOrReplace()


def run(spark: SparkSession) -> None:
    load(transform(extract(spark)))


if __name__ == "__main__":

    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    run(spark)  # add start and end time for incremental pipelines

* Let's go over the pattern used here: 

  - For the extract, we read the bronze advertiser table. Note that the bronze table is also a snapshot.
  - For the transform, we enrich with the following columns.
    1. is_active as a boolean indicating if status = active
    2. Renaming name to `advertiser_name`
  - For the load, we use createOrReplace to create a snapshot dimension

#### Exercise [15 min]

* Using a similar pattern, lets create an snapshot dimension table for `dim_campaign` 

* The dim_campaign table is created by joining bronze.campaign and bronze.ad_group tables. 

* The campaing to ad_group is a 1:many relationship (use data modeling gold chapter to model this).

* Do the following transformations:
    1. rename:
       - campaign.name to campaign_name
       - ad_group.name to ad_group_name
    3. Enrich:
        - `campaign_duration_days` = campaign.end_date - campaign.start_date
        - `is_active` (boolean) for campaign = current date >= start_date and current_date <= end_date
        - `is_active` (boolean) for ad group = adgroup.status == active
        - `is_cpc_bidding` = ad_group.bid_strategy is in ['cpc', 'manual_cpc'] 

* Solution at [dim_campaign.py](./capstone_project/silver/dim_campaign.py)

* Let's go over how to create fact tables

* Fact tables are generally much larger compared to dimension tables

* In addition they are also source from data that is usually append only (ie no updates)

* Due to these factors (size and append only) we can use incremental pattern to reduce expenses (cost and time of processing)

#### Example

* Let's look at how we can create pipeline for `fct_ad_impressions` table below

In [ ]:
# fct_ad_impressions.py
import argparse

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.silver.fct_ad_impressions"


def extract(
    spark: SparkSession, start_time: str, end_time: str
) -> dict[str, DataFrame]:
    impressions_df = spark.table("local.bronze.ad_impression").filter(
        (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
    )
    ad_creative_df = spark.table("local.bronze.ad_creative")
    return {
        "ad_impression": impressions_df,
        "ad_creative": ad_creative_df,
    }


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    ad_impression_df = input_dfs["ad_impression"]
    ad_creative_df = input_dfs["ad_creative"]

    return (
        ad_impression_df.alias("ai")
        .join(ad_creative_df.alias("ac"), on="creative_id", how="left")
        .select(
            F.col("ai.impression_id"),
            F.col("ai.creative_id"),
            F.col("ac.ad_group_id"),
            F.col("ai.session_id"),
            F.col("ai.customer_id"),
            F.col("ai.placement"),
            F.col("ai.cost"),
            F.col("ai.impressed_at"),
            F.col("ai.created_at"),
            (F.col("ai.cost") * 100).cast("long").alias("cost_in_cents"),
        )
    )


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        (
            output_df.writeTo(TABLE_NAME)
            .partitionedBy(F.partitioning.days("created_at"))
            .createOrReplace()
        )
    else:
        output_df.writeTo(TABLE_NAME).overwritePartitions()


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    load(transform(extract(spark, start_time, end_time)), spark)
    spark.stop()


"""
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=f"{TABLE_NAME} ETL")
    parser.add_argument(
        "--start-time",
        required=True,
        help="Start time (inclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    parser.add_argument(
        "--end-time",
        required=True,
        help="End time (exclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    args = parser.parse_args()

    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    run(spark, args.start_time, args.end_time)
"""

* Let's go over the pattern used here: 

  - For the extract, we read
     - The entire `bronze.ad_creative` table. Since ad_creative is a snapshot, we select the entire table.
     - We filter by time range on the fact `broze.ad_impression` table.
  - For the transform, we left join ad_creative (dimension) to ad_impression (fact) and create a column `cost_in_cents`.
  - For the load, we create a partitioned table if the destination table does not exist. If it does exist, we use `overwritePartitions()`.

* The `overwritePartitions()` + append only nature of the source fact table is critical, as it ensures that even if you reprocess data for the same created_at days the output will not change.

* Rerunning pipeline with `overwritePartitions` also handles late arriving events.

* If you re-run the pipeline for the past n days, it will capture all the data that occured in those days.

* Data teams run pipelines like these hourly and for an entire day at night to "catchup" any later arriving events. (aka Lambda architecture)

#### Exercise [15 min]
* Create pipeline scripts for `fct_ad_clicks` and `fct_ad_conversions` tables.
* You only need to use `bronze.ad_clicks` and `bronze.ad_conversions` as source for these tables respectively.
* Enrichments
  - For `fct_ad_clicks` create a cost_in_cents column
  - For `fct_ad_conversions` create a **revenue_in_cents** as revenue * 1000

In [ ]:
%%bash 
%%capture
uv run ./capstone_project/silver/dim_advertiser.py
uv run ./capstone_project/silver/dim_campaign.py
uv run ./capstone_project/silver/fct_ad_impressions.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"
uv run ./capstone_project/silver/fct_ad_clicks.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"
uv run ./capstone_project/silver/fct_ad_conversions.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"

In [ ]:
for table_name in [
    "dim_advertiser",
    "dim_campaign",
    "fct_ad_impressions",
    "fct_ad_clicks",
    "fct_ad_conversions",
]:
    display(spark.table(f"local.silver.{table_name}").limit(2).toPandas())

### Gold Pipeline Scripts

* As we discussed in the multi-hop section, the gold layer has
  * **OBT**: One big table per fact. All applicable dimensions are left joined to the key fact table.
  * **Summary**: A summary table is an OBT aggregated to the necessary grains

#### Example

* Let’s create an OBT table for ad data, based on the `silver.fct_ad_impressions` table joined with all the higher-level grain (click, conversion) and its dimensions.
* We also enrich them with the following columns:
  * is_clicked: impression resulted in a click
  * is_converted: click resulted in a conversion
  * total_cost: impression_cost + click_cost
  * roas: conversion_revenue / total_cost, null if no conversion

In [ ]:
import argparse

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.gold.obt_ad_impressions"


def extract(
    spark: SparkSession, start_time: str, end_time: str
) -> dict[str, DataFrame]:
    impressions_df = spark.table("local.silver.fct_ad_impressions").filter(
        (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
    )
    clicks_df = spark.table("local.silver.fct_ad_clicks").filter(
        (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
    )
    conversions_df = spark.table("local.silver.fct_ad_conversions").filter(
        (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
    )
    return {
        "impressions": impressions_df,
        "clicks": clicks_df,
        "conversions": conversions_df,
        "dim_advertiser": spark.table("local.silver.dim_advertiser"),
        "dim_campaign": spark.table(
            "local.silver.dim_campaign"
        ),  # includes ad_groups array
    }


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    """
    Build wide OBT at impression grain by joining:
      - fct_ad_impressions (base, already has ad_group_id from Silver)
      - fct_ad_clicks (LEFT JOIN on impression_id)
      - fct_ad_conversions (LEFT JOIN on click_id)
      - dim_campaign exploded on ad_groups array to recover ad_group-level fields
      - dim_advertiser (via advertiser_id)

    Enrichment:
      - is_clicked: impression resulted in a click
      - is_converted: click resulted in a conversion
      - total_cost: impression_cost + click_cost
    """
    impressions = input_dfs["impressions"].select(
        F.col("impression_id"),
        F.col("creative_id"),
        F.col("ad_group_id"),
        F.col("session_id"),
        F.col("customer_id"),
        F.col("placement"),
        F.col("cost").alias("impression_cost"),
        F.col("cost_in_cents").alias("impression_cost_in_cents"),
        F.col("impressed_at"),
        F.col("created_at"),
    )
    clicks = input_dfs["clicks"].select(
        F.col("click_id"),
        F.col("impression_id"),
        F.col("cost").alias("click_cost"),
        F.col("cost_in_cents").alias("click_cost_in_cents"),
        F.col("clicked_at"),
    )
    conversions = input_dfs["conversions"].select(
        F.col("conversion_id"),
        F.col("click_id"),
        F.col("order_id"),
        F.col("revenue").alias("conversion_revenue"),
        F.col("revenue_in_cents").alias("conversion_revenue_in_cents"),
        F.col("attribution"),
        F.col("converted_at"),
    )

    # Explode ad_groups array to get one row per ad_group, then select fields from struct
    dim_campaign_exploded = (
        input_dfs["dim_campaign"]
        .select(
            F.col("campaign_id"),
            F.col("advertiser_id"),
            F.col("campaign_name"),
            F.col("objective"),
            F.col("budget_total"),
            F.col("budget_daily"),
            F.col("start_date"),
            F.col("end_date"),
            F.col("is_active").alias("campaign_is_active"),
            F.col("campaign_duration_days"),
            F.explode("ad_groups").alias("ag"),
        )
        .select(
            F.col("campaign_id"),
            F.col("advertiser_id"),
            F.col("campaign_name"),
            F.col("objective"),
            F.col("budget_total"),
            F.col("budget_daily"),
            F.col("start_date"),
            F.col("end_date"),
            F.col("campaign_is_active"),
            F.col("campaign_duration_days"),
            # unpack ad_group struct fields
            F.col("ag.ad_group_id").alias("ad_group_id"),
            F.col("ag.ad_group_name").alias("ad_group_name"),
            F.col("ag.targeting_type").alias("targeting_type"),
            F.col("ag.bid_strategy").alias("bid_strategy"),
            F.col("ag.max_cpc").alias("max_cpc"),
            F.col("ag.is_active").alias("ad_group_is_active"),
            F.col("ag.is_cpc_bidding").alias("is_cpc_bidding"),
        )
    )

    dim_advertiser = input_dfs["dim_advertiser"].select(
        F.col("advertiser_id"),
        F.col("advertiser_name"),
        F.col("is_active").alias("advertiser_is_active"),
    )

    return (
        impressions.join(clicks, on="impression_id", how="left")
        .join(conversions, on="click_id", how="left")
        .join(dim_campaign_exploded, on="ad_group_id", how="left")
        .join(dim_advertiser, on="advertiser_id", how="left")
        .select(
            # keys
            F.col("impression_id"),
            F.col("click_id"),
            F.col("conversion_id"),
            F.col("session_id"),
            F.col("customer_id"),
            F.col("creative_id"),
            F.col("ad_group_id"),
            F.col("campaign_id"),
            F.col("advertiser_id"),
            F.col("order_id"),
            # timestamps
            F.col("impressed_at"),
            F.col("clicked_at"),
            F.col("converted_at"),
            # impression metrics
            F.col("placement"),
            F.col("impression_cost"),
            F.col("impression_cost_in_cents"),
            # click metrics
            F.col("click_cost"),
            F.col("click_cost_in_cents"),
            # conversion metrics
            F.col("conversion_revenue"),
            F.col("conversion_revenue_in_cents"),
            F.col("attribution"),
            # ad group descriptors
            F.col("ad_group_name"),
            F.col("targeting_type"),
            F.col("bid_strategy"),
            F.col("max_cpc"),
            F.col("ad_group_is_active"),
            F.col("is_cpc_bidding"),
            # campaign descriptors
            F.col("campaign_name"),
            F.col("objective"),
            F.col("budget_total"),
            F.col("budget_daily"),
            F.col("start_date"),
            F.col("end_date"),
            F.col("campaign_is_active"),
            F.col("campaign_duration_days"),
            # advertiser descriptors
            F.col("advertiser_name"),
            F.col("advertiser_is_active"),
            # enrichment
            F.col("click_id").isNotNull().alias("is_clicked"),
            F.col("conversion_id").isNotNull().alias("is_converted"),
            (
                F.coalesce(F.col("impression_cost"), F.lit(0))
                + F.coalesce(F.col("click_cost"), F.lit(0))
            ).alias("total_cost"),
            # partition col
            F.col("created_at"),
        )
    )


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        (
            output_df.writeTo(TABLE_NAME)
            .partitionedBy(F.partitioning.days("created_at"))
            .createOrReplace()
        )
    else:
        output_df.writeTo(TABLE_NAME).overwritePartitions()


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    load(transform(extract(spark, start_time, end_time)), spark)

"""
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=f"{TABLE_NAME} ETL")
    parser.add_argument(
        "--start-time",
        required=True,
        help="Start time (inclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    parser.add_argument(
        "--end-time",
        required=True,
        help="End time (exclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    args = parser.parse_args()

    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    run(spark, args.start_time, args.end_time)
"""

* Let's go over the pattern used here: 

  - For the **extract**, we read
    - Impression, clicks, & conversion silver fact tables with time range filter
    - Full snapshot dimensions: dim_advertiser & dim_campaign
  - For the **transform**
    - We left join higher grains (clicks and converstions) and dim_advertiser and dim_campaign to the fact table that dictates the grain of this OBT, `fct_ad_impressions`
    - We add the following enrichment columns
      - is_clicked: impression resulted in a click
      - is_converted: click resulted in a conversion
      - total_cost: impression_cost + click_cost
      - roas: conversion_revenue / total_cost, null if no conversion
  - For the **load**, we create a partitioned table if the destination table does not exist. If it does exist, we use `overwritePartitions()`.


#### Example
* Let's go over how to create gold summary table `ad_campaign_performance`, at date and campaign grain.
* We enrich with the following columns
  - total_impressions: COUNT(impression_id)
  - total_clicks: COUNT(click_id)
  - total_conversions: COUNT(conversion_id)
  - total_spend: SUM(impression_cost) + SUM(click_cost)
  - total_revenue: SUM(conversion_revenue)
  - roas: total_revenue / total_spend
  - Rolling windows (l7d, l30d, l90d) on roas.

In [ ]:
import argparse
import logging
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from soda_core.contracts import verify_contract_locally
from soda_sparkdf import SparkDataFrameDataSource

TABLE_NAME = "local.gold.ad_campaign_performance"
logger = logging.getLogger(__name__)


def extract(
    spark: SparkSession, start_time: str, end_time: str
) -> dict[str, DataFrame]:
    return {
        "obt_ad": spark.table("local.gold.obt_ad_impressions").filter(
            (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
        )
    }


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    """
    Aggregate obt_ad to campaign × day grain.
    Metrics:
      - total_impressions: COUNT(impression_id)
      - total_clicks: COUNT(click_id)
      - total_conversions: COUNT(conversion_id)
      - total_spend: SUM(impression_cost) + SUM(click_cost)
      - total_revenue: SUM(conversion_revenue)
      - roas: total_revenue / total_spend
    Rolling windows (l7d, l30d, l90d) on roas.
    """
    obt = input_dfs["obt_ad"]

    daily = (
        obt.groupBy(
            F.to_date(F.col("created_at")).alias("event_date"),
            F.col("campaign_id"),
            F.col("campaign_name"),
            F.col("advertiser_id"),
            F.col("advertiser_name"),
            F.col("objective"),
            F.col("budget_total"),
            F.col("budget_daily"),
            F.col("campaign_is_active"),
        )
        .agg(
            F.count("impression_id").alias("total_impressions"),
            F.count("click_id").alias("total_clicks"),
            F.count("conversion_id").alias("total_conversions"),
            (
                F.sum(F.coalesce(F.col("impression_cost"), F.lit(0)))
                + F.sum(F.coalesce(F.col("click_cost"), F.lit(0)))
            ).alias("total_spend"),
            F.sum(F.coalesce(F.col("conversion_revenue"), F.lit(0))).alias(
                "total_revenue"
            ),
        )
        .withColumn(
            "roas",
            F.when(
                F.col("total_spend") > 0, F.col("total_revenue") / F.col("total_spend")
            ).otherwise(F.lit(None)),
        )
    )

    window_7d = (
        Window.partitionBy("campaign_id")
        .orderBy(F.col("event_date").cast("timestamp").cast("long"))
        .rangeBetween(-7 * 86400, 0)
    )
    window_30d = (
        Window.partitionBy("campaign_id")
        .orderBy(F.col("event_date").cast("timestamp").cast("long"))
        .rangeBetween(-30 * 86400, 0)
    )
    window_90d = (
        Window.partitionBy("campaign_id")
        .orderBy(F.col("event_date").cast("timestamp").cast("long"))
        .rangeBetween(-90 * 86400, 0)
    )

    return daily.select(
        F.col("event_date"),
        F.col("campaign_id"),
        F.col("campaign_name"),
        F.col("advertiser_id"),
        F.col("advertiser_name"),
        F.col("objective"),
        F.col("budget_total"),
        F.col("budget_daily"),
        F.col("campaign_is_active"),
        F.col("total_impressions"),
        F.col("total_clicks"),
        F.col("total_conversions"),
        F.col("total_spend"),
        F.col("total_revenue"),
        F.col("roas"),
        F.sum("total_revenue").over(window_7d).alias("l7d_revenue"),
        F.sum("total_spend").over(window_7d).alias("l7d_spend"),
        (
            F.sum("total_revenue").over(window_7d)
            / F.nullif(F.sum("total_spend").over(window_7d), F.lit(0))
        ).alias("l7d_roas"),
        (
            F.sum("total_revenue").over(window_30d)
            / F.nullif(F.sum("total_spend").over(window_30d), F.lit(0))
        ).alias("l30d_roas"),
        (
            F.sum("total_revenue").over(window_90d)
            / F.nullif(F.sum("total_spend").over(window_90d), F.lit(0))
        ).alias("l90d_roas"),
    )


def load(output_df: DataFrame) -> None:
    output_df.writeTo(TABLE_NAME).createOrReplace()


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    load(transform(extract(spark, start_time, end_time)))


"""
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=f"{TABLE_NAME} ETL")
    parser.add_argument(
        "--start-time",
        required=True,
        help="Start time (inclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    parser.add_argument(
        "--end-time",
        required=True,
        help="End time (exclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    args = parser.parse_args()

    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    run(spark, args.start_time, args.end_time)
"""

* Let's go over the pattern used here: 

  - For the **extract**, we read
    - only the corresponding obt, `gold.obt_ad_impressions` with time range filter
  - For the **transform**
    - We aggregate the OBT to the desired grain: `day x campaign`
    - We enrich with the following columns:
      - total_impressions: COUNT(impression_id)
      - total_clicks: COUNT(click_id)
      - total_conversions: COUNT(conversion_id)
      - total_spend: SUM(impression_cost) + SUM(click_cost)
      - total_revenue: SUM(conversion_revenue)
      - roas: total_revenue / total_spend
      - Rolling windows (l7d, l30d, l90d) on roas.
  - For the **load**, we create a partitioned table if the destination table does not exist. If it does exist, we use `overwritePartitions()`.

* The benefit of having an OBT, we only need to aggregate.
* The joins/enrichment are done at the OBT layer
* We use the lness technique to show lastnday metrics with window functions.

#### Exercise [30 min]

* Write code to create `gold.ad_funnel_summary` following the pattern above
* Grain is `event_date x ad_group`
* With the following columns

**Dimensions**

| Column | Description |
|---|---|
| `event_date` | Date of the ad activity (truncated from `created_at`) |
| `ad_group_id` | Unique identifier for the ad group |
| `ad_group_name` | Display name of the ad group |
| `campaign_id` | Unique identifier for the campaign |
| `campaign_name` | Display name of the campaign |
| `advertiser_id` | Unique identifier for the advertiser |
| `advertiser_name` | Display name of the advertiser |
| `bid_strategy` | Bidding strategy used (e.g. CPC, CPM) |
| `ad_group_is_active` | Whether the ad group is currently active |

**Facts & Metrics**

| Column | Formula | Description |
|---|---|---|
| `total_impressions` | `COUNT(impression_id)` | Total number of times ads were displayed |
| `total_clicks` | `COUNT(click_id)` | Total number of clicks on ads |
| `total_conversions` | `COUNT(conversion_id)` | Total number of conversion events |
| `total_spend` | `SUM(impression_cost) + SUM(click_cost)` | Total amount spent across impression and click costs |

**Computed Metrics**

| Column | Formula | Description |
|---|---|---|
| `ctr` | `total_clicks / total_impressions` | Click-through rate — share of impressions that resulted in a click. Null if no impressions. |
| `cvr` | `total_conversions / total_clicks` | Conversion rate — share of clicks that resulted in a conversion. Null if no clicks. |
| `cpa` | `total_spend / total_conversions` | Cost per acquisition — average spend to generate one conversion. Null if no conversions. |
| `l7d_cvr` | `SUM(conversions, 7d) / SUM(clicks, 7d)` | Rolling 7-day CVR per ad group, smoothing out daily noise |
| `l30d_cvr` | `SUM(conversions, 30d) / SUM(clicks, 30d)` | Rolling 30-day CVR per ad group, for medium-term trend analysis |
| `l90d_cvr` | `SUM(conversions, 90d) / SUM(clicks, 90d)` | Rolling 90-day CVR per ad group, for long-term performance baseline |

In [ ]:
%%bash 
%%capture
uv run ./capstone_project/gold/obt_ad_impressions.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"
uv run ./capstone_project/gold/ad_funnel_summary.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"
uv run ./capstone_project/gold/ad_campaign_performance.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"

In [ ]:
for table_name in [
    "obt_ad_impressions",
    "ad_funnel_summary",
    "ad_campaign_performance",
]:
    display(spark.table(f"local.gold.{table_name}").limit(2).toPandas())

## Data quality

* We have the pipeline scripts, however we need to ensure that the data we present our stakeholders are correct

#### Example 

* Let's create data quality checks as part of our pipeline following the WAP pattern
* Let's look at an example of creating DQ checks for `ad_campaign_performance` summary table
  - row_count > 0
  - no nulls on campaign id and event_date
  - total_spend > 0
  - roas >= 0
* We need to create a validate function and raise an exception when there is a data quality issue

In [ ]:
import argparse
import logging
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from soda_core.contracts import verify_contract_locally
from soda_sparkdf import SparkDataFrameDataSource


def validate(output_df: DataFrame, spark: SparkSession) -> bool:
    """
    WAP pattern — validate before load using Soda Core contracts.
    Registers output_df as a timestamped temp view for debuggability.
    Checks:
      - row_count > 0
      - no nulls on campaign_id, event_date
      - total_spend >= 0
      - roas >= 0
    Returns False and logs errors on failure.
    Caller (run()) is responsible for raising exception.
    """
    view_name = "tmp_ad_campaign_perf"
    output_df.createOrReplaceTempView(view_name)

    spark_data_source = SparkDataFrameDataSource.from_existing_session(
        session=spark, name="my_sparkdf"
    )

    file_name = TABLE_NAME.split(".")[-1]
    CONTRACT_PATH = Path(__file__).parent / "contracts" / f"{file_name}.yaml"

    result = verify_contract_locally(
        data_sources=[spark_data_source],
        contract_file_path=str(CONTRACT_PATH),
    )

    if result.is_ok:
        logger.info(f"✅ Validation passed for {view_name}")
        return True
    else:
        logger.error(f"❌ Validation failed for {view_name}")
        logger.error(result.get_errors_str())
        return False


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    output_df = transform(extract(spark, start_time, end_time))
    if not validate(output_df, spark):
        raise Exception(f"Validation failed for {TABLE_NAME}, aborting load")
    load(output_df)

[./capstone_project/gold/contracts/ad_campaign_performance.yaml](./capstone_project/gold/contracts/ad_campaign_performance.yaml)

* We can see how our DQ checks are defined in the yaml and accessed by our validationfunction at pipeline runtime.
* The validation function should only tell you pass/fail
* It is upon the caller (the run function) to decide what to do with it
* In your run function will raise an exception if it notices a data quality issue
```python
    output_df = transform(extract(spark, end_time))
    if not validate(output_df, spark):
        raise Exception(f"Validation failed for {TABLE_NAME}, aborting load")
    load(output_df, spark)
```
* If there is a data quality issue no data is loaded into the stakeholder accessible table.

#### Exercise [15 min]
* Create and implement DQ checks for `ad_funnel_summary` table.
  - row_count > 0
  - no nulls on ad_group_id
  - no nulls on event_date
  - total_spend >= 0
  - cvr between 0 and 1

* We use Soda-core, here is its [docs for reference](https://docs.soda.io/reference/contract-language-reference)

* Solution at [ad_funnel_summary.py](./capstone_project/gold/ad_funnel_summary.py) & [./capstone_project/gold/contracts/ad_funnel_summary.yaml](./capstone_project/gold/contracts/ad_funnel_summary.yaml)

In [ ]:
%%bash 
%%capture
uv run ./capstone_project/gold/ad_funnel_summary.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"
uv run ./capstone_project/gold/ad_campaign_performance.py --start-time "2025-01-01 00:00:00" --end-time "2026-01-01 00:00:00"

In [ ]:
for table_name in [
    "obt_ad_impressions",
    "ad_funnel_summary",
    "ad_campaign_performance",
]:
    display(spark.table(f"local.gold.{table_name}").limit(2).toPandas())

## Visualizing outputs

* Potential employers looking at your portfolio don't spend a lot of time understanding your code base
* To show outcomes and catch attention an easy way is to create engaging visuals
* Here is a script to create visual pngs that you can attach to your README.md 

In [ ]:
! uv run ./capstone_project/charts/generate_charts.py

In [ ]:
%%bash
mv funnel.png ./capstone_project
mv roas_over_time.png ./capstone_project

![Funnel Summary](capstone_project/funnel.png)
![ROAS](capstone_project/roas_over_time.png)

[Vega-Altair Example Reference](https://vega.github.io/vega-lite/examples/)

## Orchestrate your pipelines

* In production most companies try to follow 1 pipeline 1 table policy.
* In our use case we can simplify by creating one pipeline per layer

#### Example

* Let's create a single pipeline to run all our bronze table scripts as shown below
* See [bronze_layer.py](../../airflow/dags/capstone_project/bronze_layer.py)

#### Exercise [30 min]

* Create 2 DAG but called `capstone_silver_layer` and `capstone_gold_layer`
* They should run their corresponding tables yearly.
* In the `capstone_gold_layer` ensure that the OBT table is created first before the summary tables.
* Solution [silver_layer.py](../../airflow/dags/capstone_project/silver_layer.py) & [gold_layer.py](../../airflow/dags/capstone_project/gold_layer.py)

In [ ]:
! echo $AIRFLOW_HOME/dags

## Presentation matters

* Make it easy for people to understand your expertise and your sense of e2e ownership
* This is easily done with a README
* Use the following sections to really concentrate on showcasing your expertise
* Most people just scan and do not go into details, so show key skils up to as shown below
* Lucklily the order with which we developed this capstone project works great when showcasing your expertise.

```text
# Project Title
## Stakeholders & Outcomes
## Results
## Architecture
## Code Pattern
## Data Quality
```

See [README.md](./capstone_project/README.md)

## Recap

We covered a lot in this section. We went over the following.

1. How to design a portfolio project to demonstrate expertise
2. How to architect a pipeline following industry standard patterns
3. How to quickly spin up pipeline scripts for tables
4. Implementing data quality checks
5. Visualizing outcomes
6. Orchestrating pipelines
7. Presenting with the objective of expertise demonstration